In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors

import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC


In [ ]:
# 2. Create TensorDataset objects

# Training dataset
train_dataset = CIFAR10(root='./datasets', train=True, transform=to_tensor, download=True)
# Testing dataset
test_dataset = CIFAR10(root='./datasets', train=False, transform=to_tensor, download=True)

# Print one sample from the dataset
sample_image, sample_label = train_dataset[0]
print(f"Image shape: {sample_image.shape}")  # (3, 32, 32)
print(f"Label: {sample_label}")


In [ ]:
# 3. Create DataLoaders

# DataLoader for training
 datatrain_loader = DataLoader( train_dataset, batch_size=32, shuffle=True, num_workers=2)
 # DataLoader for test
 datatest_loader = DataLoader( test_dataset, batch_size=32, shuffle=False, num_workers=2)
 # Get the first batch from the training
  DataLoaderX_batch, y_batch = next(iter(train_loader))print(f"Training batch input shape: {X_batch.shape}")print(f"Training batch labels shape: {y_batch.shape}")


keyboard_arrow_down

more_horiz
 Training batch input shape: torch.Size([32, 3, 32, 32])
Training batch labels shape: torch.Size([32])



In [ ]:
# 4. Print shape of one batch
X_batch, y_batch = next(iter(train_loader))print(f"Training batch input shape: {X_batch.shape}")print(f"Training batch labels shape: {y_batch.shape}")


keyboard_arrow_down

more_horiz
 Training batch input shape: torch.Size([32, 3, 32, 32])
Training batch labels shape: torch.Size([32])


In [ ]:
# 5. Display sample images
# Get one batch of images and labels
images, labels = next(iter(train_loader))

# Display the first 6 images in the batch
plt.figure(figsize=(8, 4))

for i in range(6):
    plt.subplot(2, 3, i + 1)

    # Convert from (C, H, W) to (H, W, C) for matplotlib
    img = images[i].permute(1, 2, 0)

    plt.imshow(img)
    plt.title(f"Label: {labels[i].item()}")
    plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Task 1: Write your model class here:
class NN4Layer(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim):
        super(NN4Layer, self).__init__()

        # TODO: What are the dimensions of the first layer? (hint: features? hidden neurons?)
        self.layer1 = nn.Linear(input_dim, hidden_dim)

        # TODO: What are the dimensions from previous and next layer?
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)

        # TODO: What are the dimensions from previous and next layer?
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)

        # TODO: What are the dimensions from previous and next layer?
        self.layer4 = nn.Linear(hidden_dim, hidden_dim)

        # activation function for non-linearity
        self.relu = nn.ReLU()


In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, optimizer, criterion, train_loader, device):
    # Set the model to training mode
    model.train()

    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        # Move batch to the selected device
        X_batch = X_batch.view(X_batch.size(0), -1).to(device)
        y_batch = y_batch.to(device)

        # Forward pass
        # TODO: make predictions
        outputs = model(X_batch) # shape: (batch_size, 10)
        # TODO: compute loss
        loss = criterion(outputs, y_batch)

        # Backward pass & optimization
        optimizer.zero_grad()   # Clear previous gradients
        loss.backward()         # Compute gradients
        optimizer.step()        # Update model parameters

        running_loss += loss.item()

    # Average loss over all batches
    avg_loss = running_loss / len(train_loader)

    return avg_loss

In [ ]:
# Task 3: Write your validation loop here:
def validate(model, criterion, test_loader, device):
    # Set the model to evaluation mode
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            # Move data to device
            X_batch = X_batch.view(X_batch.size(0), -1).to(device)
            y_batch = y_batch.to(device)

            # Forward pass
            # TODO: make predictions
            outputs = model(X_batch)  # shape: (batch_size, 10)
            # TODO: compute loss
            loss = criterion(outputs, y_batch)
            running_loss += loss.item()

            # TODO: Apply Softmax to get probabilities
            probabilities = F.softmax(outputs, dim=1)

            # TODO: Multiclass predictions
            predicted = torch.argmax(probabilities, dim=1)

            # Accuracy calculation
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)

    avg_loss = running_loss / len(test_loader)
    # TODO: how do we calculate accuracy?
    accuracy = correct / total

    return avg_loss, accuracy

In [ ]:
# Task 4: Define device, model, loss, optimizer:
# Scatter plot: Predicted vs Actual
plt.figure(figsize=(8, 6))

# Plot: Predicted vs Actual scatter
plt.scatter(y_test.numpy(), predictions.flatten(), alpha=0.5, s=10, c='steelblue')

# Add perfect prediction line
min_val = min(y_test.min(), predictions.min())
max_val = max(y_test.max(), predictions.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')

plt.xlabel('Actual Price ($)', fontsize=12)
plt.ylabel('Predicted Price ($)', fontsize=12)
plt.title('Predicted vs Actual Diamond Prices', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Task 1: Write your code here:

Plotting resultsplt.figure(figsize=(7, 5))plt.plot(train_losses, label='Train Loss')plt.plot(val_losses, label='Validation Loss')plt.title('Loss over Epochs')plt.xlabel('Epoch')plt.ylabel('Loss')plt.legend()plt.tight_layout()plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here:

In [ ]:
# Task 5: Start training for 20 epochs:
print (ANON)
